# 09 Reranking

Το notebook εφαρμόζει cross-encoder reranking στα hybrid retrieval candidates. Στόχος είναι να βελτιωθεί η τελική σειρά των ανακτημένων αποσπασμάτων πριν από το στάδιο παραγωγής απαντήσεων.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from sentence_transformers import CrossEncoder
from tqdm import tqdm

from scripts.reranking_utils import (
    build_reranker_document,
    fuse_rerank_scores,
    validate_rerank_candidate_pool,
)

In [ ]:
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"

RERANK_TOP_N = 20
FINAL_TOP_K = 5
RERANK_SCORE_WEIGHT = 0.7
HYBRID_SCORE_WEIGHT = 0.3

USE_QUERY_LIMIT = False
QUERY_LIMIT = 50

rerank_config = {
    "retrieval_type": "hybrid_reranked",
    "rerank_model": RERANK_MODEL,
    "rerank_top_n": RERANK_TOP_N,
    "final_top_k": FINAL_TOP_K,
    "generation_context_k": FINAL_TOP_K,
    "rerank_score_weight": RERANK_SCORE_WEIGHT,
    "hybrid_score_weight": HYBRID_SCORE_WEIGHT,
    "metadata_aware": True,
    "document_known": False,
    "candidate_source": "hybrid"
}

rerank_config

In [ ]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RETRIEVAL_DIR = PROCESSED_DIR / "retrieval_results"

HYBRID_RESULTS_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid.csv"
HYBRID_RESULTS_PARQUET_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid.parquet"

RERANK_RESULTS_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid_reranked.csv"
RERANK_RESULTS_PARQUET_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid_reranked.parquet"
RERANK_MANIFEST_PATH = RETRIEVAL_DIR / "retrieval_manifest_hybrid_reranked.csv"
RERANK_STATS_PATH = RETRIEVAL_DIR / "retrieval_stats_hybrid_reranked.json"
RERANK_CANDIDATES_CSV_PATH = RETRIEVAL_DIR / "retrieval_candidates_hybrid_reranked.csv"
RERANK_CANDIDATES_PARQUET_PATH = RETRIEVAL_DIR / "retrieval_candidates_hybrid_reranked.parquet"

print("RETRIEVAL_DIR:", RETRIEVAL_DIR)

In [ ]:
if HYBRID_RESULTS_PARQUET_PATH.exists():
    hybrid_df = pd.read_parquet(HYBRID_RESULTS_PARQUET_PATH)
elif HYBRID_RESULTS_CSV_PATH.exists():
    hybrid_df = pd.read_csv(HYBRID_RESULTS_CSV_PATH)
else:
    raise FileNotFoundError("Hybrid retrieval results not found.")

print("hybrid_df shape:", hybrid_df.shape)
print(hybrid_df.columns.tolist())
hybrid_df.head(2)

In [ ]:
required_cols = [
    "financebench_id",
    "question",
    "question_clean",
    "expanded_question",
    "retrieved_rank",
    "chunk_id",
    "retrieved_doc_id",
    "chunk_text",
]

missing_cols = [c for c in required_cols if c not in hybrid_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in hybrid_df: {missing_cols}")

print("Hybrid retrieval columns OK.")

In [ ]:
if USE_QUERY_LIMIT:
    keep_ids = hybrid_df["financebench_id"].drop_duplicates().head(QUERY_LIMIT).tolist()
    hybrid_df = hybrid_df[hybrid_df["financebench_id"].isin(keep_ids)].copy().reset_index(drop=True)

print("hybrid_df shape after optional limit:", hybrid_df.shape)
print("n unique queries:", hybrid_df["financebench_id"].nunique())

In [ ]:
validate_rerank_candidate_pool(hybrid_df, top_n=RERANK_TOP_N)
print(
    f"Candidate pool validated: at least {RERANK_TOP_N} unique rows per query."
)


In [ ]:
reranker = CrossEncoder(RERANK_MODEL)
print("Loaded reranker:", RERANK_MODEL)

In [ ]:
def rerank_candidates(question: str, candidate_df: pd.DataFrame, top_n: int = RERANK_TOP_N):
    candidate_df = (
        candidate_df.sort_values("retrieved_rank", ascending=True)
        .head(top_n)
        .copy()
    )

    # Only metadata attached to retrieved chunks is used. Gold document/company
    # fields are deliberately excluded to prevent evaluation leakage.
    candidate_df["rerank_input_text"] = candidate_df.apply(
        build_reranker_document, axis=1
    )
    pairs = list(zip(
        [question] * len(candidate_df),
        candidate_df["rerank_input_text"].tolist(),
    ))
    candidate_df["rerank_score"] = reranker.predict(pairs)

    return fuse_rerank_scores(
        candidate_df,
        rerank_weight=RERANK_SCORE_WEIGHT,
        hybrid_weight=HYBRID_SCORE_WEIGHT,
    )

In [ ]:
reranked_records = []
candidate_audit_frames = []
manifest_records = []

grouped = hybrid_df.groupby("financebench_id", sort=False)

for financebench_id, group in tqdm(grouped, total=hybrid_df["financebench_id"].nunique(), desc="Running reranking"):
    first_row = group.iloc[0]

    question = first_row["question"]
    question_clean = first_row["question_clean"]
    expanded_question = first_row["expanded_question"]
    expected_doc_name = first_row.get("expected_doc_name")
    expected_company = first_row.get("expected_company")

    manifest_record = {
        "financebench_id": financebench_id,
        "question": question,
        "expected_doc_name": expected_doc_name,
        "status": None,
        "error_message": None,
        "n_input_candidates": int(len(group)),
        "n_reranked_results": 0
    }

    try:
        scored_group = rerank_candidates(
            question=question_clean,
            candidate_df=group,
            top_n=min(RERANK_TOP_N, len(group))
        )

        audit_group = scored_group.copy()
        audit_group["selected_for_generation"] = (
            audit_group["fused_rank"] <= FINAL_TOP_K
        )
        candidate_audit_frames.append(audit_group)

        reranked_group = scored_group.head(FINAL_TOP_K).copy().reset_index(drop=True)

        for new_rank, (_, row) in enumerate(reranked_group.iterrows(), start=1):
            reranked_records.append({
                "financebench_id": financebench_id,
                "question": question,
                "question_clean": question_clean,
                "expanded_question": expanded_question,
                "expected_doc_name": expected_doc_name,
                "expected_company": expected_company,
                "retrieved_rank": new_rank,
                "rerank_score": float(row["rerank_score"]),
                "rerank_score_normalized": float(row["rerank_score_normalized"]),
                "rrf_score_normalized": float(row["rrf_score_normalized"]),
                "fused_score": float(row["fused_score"]),
                "pure_rerank_rank": int(row["pure_rerank_rank"]),
                "fused_rank": int(row["fused_rank"]),
                "previous_rank": row["retrieved_rank"],
                "rrf_score": row.get("rrf_score"),
                "dense_rank": row.get("dense_rank"),
                "dense_score": row.get("dense_score"),
                "bm25_rank": row.get("bm25_rank"),
                "bm25_score": row.get("bm25_score"),
                "chunk_id": row["chunk_id"],
                "retrieved_doc_id": row["retrieved_doc_id"],
                "chunk_index": row.get("chunk_index"),
                "chunk_text": row["chunk_text"],
                "char_count": row.get("char_count"),
                "token_estimate": row.get("token_estimate"),
            })

        manifest_record["status"] = "success"
        manifest_record["n_reranked_results"] = int(len(reranked_group))

    except Exception as e:
        manifest_record["status"] = "error"
        manifest_record["error_message"] = str(e)

    manifest_records.append(manifest_record)

reranked_df = pd.DataFrame(reranked_records)
rerank_candidates_df = (
    pd.concat(candidate_audit_frames, ignore_index=True)
    if candidate_audit_frames
    else pd.DataFrame()
)
rerank_manifest_df = pd.DataFrame(manifest_records)

print("reranked_df shape:", reranked_df.shape)
print("rerank candidates audit shape:", rerank_candidates_df.shape)
print("rerank_manifest_df shape:", rerank_manifest_df.shape)

In [ ]:
if "expected_doc_name" not in reranked_df.columns:
    reranked_df["expected_doc_name"] = None

reranked_df["doc_match"] = (
    reranked_df["expected_doc_name"].fillna("").astype(str)
    == reranked_df["retrieved_doc_id"].fillna("").astype(str)
)

reranked_df[[
    "financebench_id",
    "retrieved_rank",
    "previous_rank",
    "retrieved_doc_id",
    "expected_doc_name",
    "doc_match",
    "rerank_score",
    "fused_score",
    "pure_rerank_rank"
]].head(15)

In [ ]:
top1_df = reranked_df[reranked_df["retrieved_rank"] == 1].copy()

top1_doc_match_rate = float(top1_df["doc_match"].mean()) if len(top1_df) else 0.0
topk_doc_match_rate = float(
    reranked_df.groupby("financebench_id")["doc_match"].max().mean()
) if len(reranked_df) else 0.0

summary_df = pd.DataFrame([{
    "n_queries": int(reranked_df["financebench_id"].nunique()),
    "top1_doc_match_rate": top1_doc_match_rate,
    f"top{FINAL_TOP_K}_doc_match_rate": topk_doc_match_rate
}])

summary_df

In [ ]:
reranked_df.to_csv(RERANK_RESULTS_CSV_PATH, index=False, encoding="utf-8")
reranked_df.to_parquet(RERANK_RESULTS_PARQUET_PATH, index=False)

rerank_candidates_df.to_csv(
    RERANK_CANDIDATES_CSV_PATH, index=False, encoding="utf-8"
)
rerank_candidates_df.to_parquet(RERANK_CANDIDATES_PARQUET_PATH, index=False)
rerank_manifest_df.to_csv(RERANK_MANIFEST_PATH, index=False, encoding="utf-8")

print("Saved reranking outputs:")
print("-", RERANK_RESULTS_CSV_PATH)
print("-", RERANK_RESULTS_PARQUET_PATH)
print("-", RERANK_CANDIDATES_CSV_PATH)
print("-", RERANK_CANDIDATES_PARQUET_PATH)
print("-", RERANK_MANIFEST_PATH)


In [ ]:
rerank_stats = {
    "retrieval_type": "hybrid_reranked",
    "document_known": False,
    "rerank_model": RERANK_MODEL,
    "rerank_top_n": RERANK_TOP_N,
    "final_top_k": FINAL_TOP_K,
    "generation_context_k": FINAL_TOP_K,
    "metadata_aware": True,
    "rerank_score_weight": RERANK_SCORE_WEIGHT,
    "hybrid_score_weight": HYBRID_SCORE_WEIGHT,
    "n_queries": int(reranked_df["financebench_id"].nunique()),
    "n_result_rows": int(len(reranked_df)),
    "n_manifest_rows": int(len(rerank_manifest_df)),
    "top1_doc_match_rate": top1_doc_match_rate,
    f"top{FINAL_TOP_K}_doc_match_rate": topk_doc_match_rate,
    "results_csv": str(RERANK_RESULTS_CSV_PATH),
    "results_parquet": str(RERANK_RESULTS_PARQUET_PATH),
    "manifest_csv": str(RERANK_MANIFEST_PATH),
    "candidate_audit_csv": str(RERANK_CANDIDATES_CSV_PATH),
}

with open(RERANK_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(rerank_stats, f, indent=2, ensure_ascii=False)

print("Saved stats:", RERANK_STATS_PATH)
rerank_stats

In [ ]:
print(rerank_manifest_df["status"].value_counts(dropna=False))
rerank_manifest_df[["financebench_id", "status", "error_message"]].head(10)

In [ ]:
reranked_df[[
    "financebench_id",
    "question",
    "retrieved_rank",
    "previous_rank",
    "retrieved_doc_id",
    "expected_doc_name",
    "doc_match",
    "rerank_score",
    "fused_score",
    "pure_rerank_rank",
    "chunk_id"
]].head(20)

## Συμπέρασμα

Σε αυτό το notebook:

- φορτώθηκαν τα hybrid retrieval candidates
- εφαρμόστηκε cross-encoder reranking
- διατηρήθηκαν τα τελικά reranked top-k results
- αποθηκεύτηκαν results, manifest και stats

Το επόμενο notebook θα χρησιμοποιήσει dense / hybrid / hybrid-reranked retrieval outputs για το QA stage.